# Getting started with PYVGC

Explore Pokémon Champions regulation data and basic damage calculations with `pyvgc`. The package supports Python 3.9+ and is currently in alpha version.

Run the cells from top to bottom. This notebook uses only the package's public API and the Python standard library.

## Install

Install the published package in the active notebook environment. If you are running this notebook from a cloned `pyvgc` repository instead, you can install the local checkout with `%pip install -e .` from the repository root.

## Load a regulation and inspect Pokémon

The package currently includes regulations M-A, M-B, and M-C. This example loads M-C and creates Incineroar and Venusaur with their regulation data.

In [1]:
from pyvgc import (
    load_regulation,
)

regulation = load_regulation("M-C")
attacker = regulation.load_pokemon("Incineroar")
attacker.moveset = ("Acrobatics", "Crunch", "Flamethrower", "Protect")
defender = regulation.load_pokemon("Venusaur")

print(attacker.info())
print(defender.info())

{'name': 'Incineroar', 'stats': {'hp': 170, 'attack': 135, 'defense': 110, 'special_attack': 100, 'special_defense': 110, 'speed': 80}, 'types': ('Fire', 'Dark'), 'nature': None, 'item': None, 'moveset': ('Acrobatics', 'Crunch', 'Flamethrower', 'Protect')}
{'name': 'Venusaur', 'stats': {'hp': 155, 'attack': 102, 'defense': 103, 'special_attack': 120, 'special_defense': 120, 'speed': 100}, 'types': ('Grass', 'Poison'), 'nature': None, 'item': None, 'moveset': ()}


## Calculate damage for damaging moves

A Pokémon's movepool contains every move it can learn in the regulation. Its moveset contains up to four equipped moves. Each moveset entry is a Move object that can be passed directly to the damage calculator.

In [2]:
damaging_moves = [
    move
    for move in attacker.moveset
    if move.base_power is not None
]

for move in damaging_moves:
    damage = regulation.battle_logic.calculate_damage(
        attacker,
        defender,
        move=move,
    )
    print(f"{attacker.name} using {move.name} on {defender.name}: {damage}")

Incineroar using Acrobatics on Venusaur: {'normal': [56, 56, 56, 58, 58, 58, 60, 60, 60, 62, 62, 62, 64, 64, 64, 66], 'critical': [82, 84, 84, 86, 86, 88, 88, 90, 90, 92, 92, 94, 94, 96, 96, 98], 'normal_percentage': [36.13, 36.13, 36.13, 37.42, 37.42, 37.42, 38.71, 38.71, 38.71, 40.0, 40.0, 40.0, 41.29, 41.29, 41.29, 42.58], 'critical_percentage': [52.9, 54.19, 54.19, 55.48, 55.48, 56.77, 56.77, 58.06, 58.06, 59.35, 59.35, 60.65, 60.65, 61.94, 61.94, 63.23]}
Incineroar using Crunch on Venusaur: {'normal': [60, 61, 61, 63, 63, 64, 64, 66, 66, 67, 67, 69, 69, 70, 70, 72], 'critical': [91, 91, 93, 94, 96, 96, 97, 99, 99, 100, 102, 103, 103, 105, 106, 108], 'normal_percentage': [38.71, 39.35, 39.35, 40.65, 40.65, 41.29, 41.29, 42.58, 42.58, 43.23, 43.23, 44.52, 44.52, 45.16, 45.16, 46.45], 'critical_percentage': [58.71, 58.71, 60.0, 60.65, 61.94, 61.94, 62.58, 63.87, 63.87, 64.52, 65.81, 66.45, 66.45, 67.74, 68.39, 69.68]}
Incineroar using Flamethrower on Venusaur: {'normal': [86, 90, 90,

## Query regulation data

Utility functions can look up regulation contents, Pokémon by ability, Pokémon that learn a move, and moves available to a Pokémon.

In [3]:
from pyvgc import (
    get_movepool_for_pokemon,
    get_pokemon_for_move,
    pokemon_with_ability,
    regulation_description,
)

print("Regulation M-A:", regulation_description("M-A"))
print("Intimidate users in M-C:", pokemon_with_ability("M-C", "Intimidate"))
print("Pokémon with Follow Me in their M-C movepool:", get_pokemon_for_move("Follow Me", regulation="M-C"))

venusaur_movepool = get_movepool_for_pokemon("Venusaur", regulation="M-C")
print("Movepool for Venusaur in M-C:")
for move in venusaur_movepool:
    print(move.name)

Regulation M-A: Pokémon: 332, Items: 119, Moves: 490
Intimidate users in M-C: ('Arbok', 'Arcanine', 'Arcanine-Hisuian', 'Gyarados', 'Incineroar', 'Krookodile', 'Luxray', 'Mabosstiff', 'Manectric-Mega', 'Mawile', 'Overqwil', 'Qwilfish', 'Salamence', 'Scrafty', 'Scrafty-Mega', 'Squawkabilly', 'Squawkabilly-Blue-Plumage', 'Squawkabilly-White-Plumage', 'Squawkabilly-Yellow-Plumage', 'Staraptor', 'Tauros', 'Tauros-Aqua', 'Tauros-Blaze', 'Tauros-Paldean', 'Wyrdeer')
Pokémon with Follow Me in their M-C movepool: ('Clefable', 'Clefable-Mega', 'Indeedee', 'Maushold', 'Maushold-Family-of-Three')
Movepool for Venusaur in M-C:
Acid Spray
Amnesia
Body Slam
Bulldoze
Bullet Seed
Charm
Curse
Double-Edge
Earth Power
Earthquake
Endure
Energy Ball
Facade
Frenzy Plant
Giga Drain
Giga Impact
Grass Knot
Grassy Glide
Grassy Terrain
Growth
Helping Hand
Hyper Beam
Ingrain
Knock Off
Leaf Storm
Leech Seed
Light Screen
Outrage
Petal Blizzard
Petal Dance
Poison Jab
Poison Powder
Power Whip
Protect
Rest
Roar
Round


In [4]:
from pyvgc import Pokemon, calculate_damage, create_custom_move

electro_drift = create_custom_move(
    "ElectroDrift",
    base_power=120,
    move_type="Electric",
    category="Special",
)
attacker = Pokemon.load("Incineroar")
target = Pokemon.load("Venusaur")
rolls = calculate_damage(attacker, target, move=electro_drift)


## You can also import and export Pokémon sets into the Showdown format. This is still under development

In [5]:
from pyvgc import import_showdown_pokemon, export_showdown_pokemon

showdown_set = """Nick (Archaludon) @ Leftovers  
        Ability: Stamina  
        Level: 50  
        EVs: 2 HP / 32 SpA / 32 Spe  
        Timid Nature  
        - Dragon Pulse  
        - Electro Shot  
        - Protect  
        - Snarl"""

pokemon = import_showdown_pokemon(showdown_set, regulation="M-C")
print("imported",pokemon.info())
datarch = export_showdown_pokemon(pokemon)
print(datarch)

custom_archaludon = Pokemon.load(   "Archaludon", 
                                    regulation="M-C",
                                    item="Leftovers", 
                                    stat_training=
                                        {"hp": 2, "attack": 0, "defense": 0, 
                                        "special_attack": 32, "special_defense": 0, "speed": 32}, 
                                    nature="Modest",
                                    moveset=("Dragon Pulse", "Electro Shot", "Protect", "Flash Cannon"),
                                    nickname="Nick",
                                    ability="Stalwart")

print(custom_archaludon.info())
print(export_showdown_pokemon(custom_archaludon))

ImportError: cannot import name 'import_showdown_pokemon' from 'pyvgc' (/Users/pablo/opt/anaconda3/envs/pyvgc_realenv/lib/python3.9/site-packages/pyvgc/__init__.py)